# Assignment 02 — Application 2: House-Price Prediction

**Student:** &lt;name&gt; &lt;id&gt;  |  **Class:** &lt;class&gt;  |  **Date:** 2026-09-03  
**Lecturer:** Dinh Que Tran, Ph.D., Assoc. Prof.  |  **Semester:** I.2026

This notebook follows **Appendix B** of the assignment: the 23 sections below take the
raw Vietnamese real-estate listings CSV through

```
Data -> Understand -> Clean -> Represent -> Learn -> Evaluate -> Persist -> Deploy
```

Every table or figure is followed by a short markdown interpretation. `RANDOM_SEED = 42`
everywhere. The persisted pipeline written in section 22 is what `../api/` serves.

## 0. Header & setup

Fix and print the environment (seed + library versions) so the experiment is reproducible.

In [ ]:
import sys, platform, random, time, warnings
import numpy as np, pandas as pd
import sklearn, matplotlib, matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

print('Python      :', sys.version.split()[0], 'on', platform.system(), platform.release())
print('numpy       :', np.__version__)
print('pandas      :', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('matplotlib  :', matplotlib.__version__)
try:
    import xgboost; print('xgboost     :', xgboost.__version__)
except ImportError:
    print('xgboost     : NOT INSTALLED  (pip install xgboost)')
print('RANDOM_SEED  =', RANDOM_SEED)

## 1. Problem definition  (markdown only)

**Real-world problem.** Property portals and buyers need a fair price estimate for a
residential listing from its attributes, so an asking price can be sanity-checked before
it is published or accepted.

**Supervised task.** **Regression** — the target is a continuous amount, so the model
minimises an error loss (MSE) and is scored with MAE / RMSE / R², **not** accuracy or a
confusion matrix. This is the key difference from the diabetes classification app.

- `X` = property attributes: area, frontage/length, room counts, floors, alley width,
  property type, position, compass direction, road type, province, and a district
  parsed from the free-text location.
- `y` = `Price`, the listing price in **million VND** (title "2.7 tỷ" ↔ `Price` = 2700).

$$ X \in \mathbb{R}^{N \times d}, \qquad y \in \mathbb{R}^{N} $$

## 2. Dataset source

| Field | Value |
|---|---|
| **Name** | VN Real Estate Listings (April–September 2025) |
| **File** | `../data/VN-real-estate-Apr-Sept-2025.csv` |
| **Rows / cols (raw)** | 236,226 × 28 |
| **Kaggle URL** | TODO — paste the exact dataset URL you downloaded from |
| **Licence / version** | TODO — e.g. CC0 / scraped-data, downloaded 2025-09 |
| **Collection** | Scraped from a Vietnamese property-listing portal; listings updated Apr–Sept 2025 |
| **Encoding / sep** | UTF-8 **with BOM** (`utf-8-sig`), comma-separated |

Full column reference and known quality issues: `../data/README.md`.

## 3. Dataset loading

In [ ]:
CSV_PATH = '../data/VN-real-estate-Apr-Sept-2025.csv'   # separator ',', encoding utf-8-sig
df = pd.read_csv(CSV_PATH, encoding='utf-8-sig', low_memory=False)
df.head()

**Interpretation.** The file loads without error. One row is one **property listing**:
its attributes plus the asking `Price`. The `Title` and `Description` columns are free
text; `Title` embeds the price and area verbatim (e.g. "Bán nhà 78.7m² 2.7 tỷ …"), so it
**must be dropped** later — using it would leak the target.

## 4. Dataset inspection

In [ ]:
print('shape (rows, columns):', df.shape)     # <-- the dataset shape
df.info()
display(df.describe(include='number').T)
print('\nmissing per column:')
print(df.isna().sum().sort_values(ascending=False))
print('\nexact duplicate rows:', df.duplicated().sum())

**Interpretation.** `df.shape = (236226, 28)` — **236,226 listings**, **28 attributes**
(1 target `Price` + 27 candidate inputs). `describe()` exposes corruption in the two most
important numeric columns: `Price` ranges from `0` to `9.2e12` and `Area` from `-6` to
`1e18` — physically impossible, so these are data-entry errors handled in section 8.
Several columns are mostly missing (`Bathrooms` ~84%, `Floors` ~79%, `Bedrooms` ~72%,
`Direction` ~70%, `Road Type` ~67%). `VIP Account` is constant. 0 exact duplicate rows,
but `Listing ID` repeats — checked in section 7.

## 5. Data-quality analysis  (issue -> count -> planned action)

In [ ]:
PRICE_LO, PRICE_HI = 0, 200_000        # million VND  (~0 .. ~200 tỷ)
AREA_LO, AREA_HI   = 0, 10_000         # m^2

rows = []
rows.append(['Price', 'value <= 0 (impossible)', int((df['Price'] <= PRICE_LO).sum()), 'drop row'])
rows.append(['Price', f'value > {PRICE_HI} (implausible outlier)', int((df['Price'] > PRICE_HI).sum()), 'drop row'])
rows.append(['Area',  'value <= 0 (impossible)', int((df['Area'] <= AREA_LO).sum()), 'drop row'])
rows.append(['Area',  f'value > {AREA_HI} (implausible outlier)', int((df['Area'] > AREA_HI).sum()), 'drop row'])
rows.append(['Area',  'missing', int(df['Area'].isna().sum()), 'drop row (target-critical size)'])
for c in ['Bedrooms', 'Bathrooms', 'Floors']:
    rows.append([c, 'missing', int(df[c].isna().sum()), 'median impute + missing-indicator flag'])
for c in ['Width', 'Length', 'Alley Width']:
    rows.append([c, 'missing', int(df[c].isna().sum()), 'median impute (in pipeline)'])
for c in ['Position', 'Direction', 'Road Type']:
    rows.append([c, 'missing', int(df[c].isna().sum()), "fill 'Unknown' category"])
rows.append(['VIP Account', 'constant column', int(df['VIP Account'].nunique()), 'drop column'])
rows.append(['Listing ID', 'duplicate ids', int(df['Listing ID'].duplicated().sum()), 'drop dupes by id (section 7)'])
rows.append(['Title', 'free text — embeds Price', df['Title'].notna().sum(), 'DROP (target leakage)'])
rows.append(['Description', 'free text', int(df['Description'].isna().sum()), 'drop column (not used for tabular model)'])
quality = pd.DataFrame(rows, columns=['column', 'issue', 'count', 'planned action'])
quality

**Explain the results.** The dominant problems are (a) **corrupted `Price` / `Area`**
— a small fraction of rows carry impossible or absurd values, dropped outright; (b)
**heavy missingness** in room-count and road columns — imputed, with a flag where the
column is >50% missing so the model can use "was this recorded?"; (c) **`VIP Account`
constant** and two **free-text columns** — removed. Actions are applied in sections 6–9
and 13–15.

## 6. Missing-value analysis  (per column decision)

In [ ]:
miss = df.isna().sum()
miss_pct = (miss / len(df) * 100).round(1)
miss_tbl = pd.DataFrame({'missing_count': miss, 'missing_pct': miss_pct})
miss_tbl = miss_tbl[miss_tbl['missing_count'] > 0].sort_values('missing_pct', ascending=False)
miss_tbl

**Strategy per column (justified).**

| Column | ~% missing | Decision | Why |
|---|---|---|---|
| `Area` | ~0.1% | **drop the row** | size is the strongest price driver; too central to impute |
| `Width` | ~26% | median impute (pipeline) | roughly symmetric; correlated with `Area` |
| `Length` | ~58% | median impute (pipeline) | high, but adds shape signal; keep column |
| `Bedrooms` | ~72% | median impute **+ `Bedrooms_missing` flag** | often absent for land listings — missingness is informative |
| `Bathrooms` | ~84% | median impute **+ `Bathrooms_missing` flag** | same |
| `Floors` | ~79% | median impute **+ `Floors_missing` flag** | same |
| `Alley Width` | ~70% | median impute (pipeline) | only meaningful for alley properties |
| `Position` / `Direction` / `Road Type` | 40–70% | fill **'Unknown'** category | absence is a category, not a number |
| `Latitude` / `Longitude` | ~68% | **drop columns** | too sparse to rely on; province + district cover location |

All imputers are **fitted inside the pipeline on the training split only** (section 15).

## 7. Duplicate analysis

In [ ]:
n_before = len(df)
print('exact duplicate rows        :', df.duplicated().sum())
print('duplicate Listing ID values :', df['Listing ID'].duplicated().sum())

# Listing ID is a natural key — the same advert re-scraped on different days.
df = df.drop_duplicates(subset='Listing ID', keep='last').reset_index(drop=True)
print(f'rows: {n_before} -> {len(df)}  (removed {n_before - len(df)} re-scrapes)')

**Interpretation.** There are no byte-identical rows, but ~19.5k listings share a
`Listing ID` — the same advert captured on more than one scrape date. Keeping all of
them would (i) inflate `N`, (ii) let the *same* property fall in both train and test,
leaking information. We keep the **last** capture per `Listing ID`. `N` drops accordingly.

## 8. Invalid-value analysis  (domain checks)

In [ ]:
checks = {
    'Price <= 0'            : (df['Price'] <= PRICE_LO).sum(),
    f'Price > {PRICE_HI}'   : (df['Price'] > PRICE_HI).sum(),
    'Area <= 0'             : (df['Area'] <= AREA_LO).sum(),
    f'Area > {AREA_HI}'     : (df['Area'] > AREA_HI).sum(),
    'Bedrooms < 0'          : (df['Bedrooms'] < 0).sum(),
    'Floors < 0'            : (df['Floors'] < 0).sum(),
}
print(pd.Series(checks, name='violations'))

n_before = len(df)
df = df[(df['Price'] > PRICE_LO) & (df['Price'] <= PRICE_HI)]
df = df[(df['Area'].notna()) & (df['Area'] > AREA_LO) & (df['Area'] <= AREA_HI)]
df = df.reset_index(drop=True)
print(f'\nrows after domain filter: {n_before} -> {len(df)}  ({len(df)/n_before*100:.1f}% kept)')

**Treatment and why.** A price of 0 (or 9e12 VND) and an area of −6 (or 1e18 m²) cannot
describe a real property — they are recording errors, not signal, and would dominate any
error-based loss. We **drop** those rows rather than cap them, because the corrupt values
give no usable information about price. The bounds (`0 < Price ≤ 200,000` million VND,
`0 < Area ≤ 10,000` m²) keep ~93% of listings. *(Diabetes analogue: `Glucose == 0`.)*

## 9. Outlier analysis

In [ ]:
num_cols_raw = ['Price', 'Area', 'Width', 'Length', 'Bedrooms', 'Bathrooms', 'Floors', 'Alley Width']
Q1, Q3 = df[num_cols_raw].quantile(0.25), df[num_cols_raw].quantile(0.75)
IQR = Q3 - Q1
iqr_outliers = ((df[num_cols_raw] < Q1 - 1.5*IQR) | (df[num_cols_raw] > Q3 + 1.5*IQR)).sum()
print('IQR-rule outlier counts:')
print(iqr_outliers)

fig, ax = plt.subplots(2, 4, figsize=(15, 6))
for a, c in zip(ax.ravel(), num_cols_raw):
    df.boxplot(column=c, ax=a); a.set_title(c)
plt.tight_layout(); plt.show()

**Decision.** `Price`, `Area`, `Width` and `Length` have long right tails, but a 15 tỷ
villa on 800 m² is a **genuine** high-end property, not an error (the impossible values
were already removed in section 8). Deleting these rows would bias the model low and
shrink coverage of the exact segment users care about. We therefore **keep** them and
instead **model `log1p(Price)`** (section 12), which compresses the tail so linear /
distance-based models are not dominated by a few large listings. Extreme `Area` / `Width`
are additionally **capped at the 99th percentile** inside feature engineering (section 13).

## 10. Exploratory data analysis  (>= 3 meaningful plots)

In [ ]:
df['price_log'] = np.log1p(df['Price'])
df['price_per_m2'] = df['Price'] / df['Area']

fig, ax = plt.subplots(1, 3, figsize=(17, 4))

# Plot 1 — target distribution (raw vs log)
ax[0].hist(df['Price'], bins=60)
ax[0].set_title('Price (million VND) — raw'); ax[0].set_xlabel('Price')

# Plot 2 — log target
ax[1].hist(df['price_log'], bins=60)
ax[1].set_title('log1p(Price) — modelling target'); ax[1].set_xlabel('log1p(Price)')

# Plot 3 — price vs area (log-log), coloured by property type count
samp = df.sample(min(8000, len(df)), random_state=RANDOM_SEED)
ax[2].scatter(np.log1p(samp['Area']), samp['price_log'], s=6, alpha=0.3)
ax[2].set_title('log Price vs log Area'); ax[2].set_xlabel('log1p(Area)'); ax[2].set_ylabel('log1p(Price)')
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 4))

# Plot 4 — median price by property type
df.groupby('Property Type')['Price'].median().sort_values().plot(kind='barh', ax=ax[0])
ax[0].set_title('Median Price by Property Type'); ax[0].set_xlabel('median Price (million VND)')

# Plot 5 — correlation heatmap of numeric features
corr_cols = ['Price', 'price_log', 'Area', 'Width', 'Length', 'Bedrooms', 'Bathrooms', 'Floors', 'Alley Width']
cm = df[corr_cols].corr()
im = ax[1].imshow(cm, cmap='coolwarm', vmin=-1, vmax=1)
ax[1].set_xticks(range(len(corr_cols))); ax[1].set_xticklabels(corr_cols, rotation=90)
ax[1].set_yticks(range(len(corr_cols))); ax[1].set_yticklabels(corr_cols)
ax[1].set_title('Correlation (numeric)'); plt.colorbar(im, ax=ax[1])
plt.tight_layout(); plt.show()

**Plot 1 & 2 — target distribution.** *Observation:* raw `Price` is extremely
right-skewed (mass near 2–5 tỷ, a thin tail to ~200 tỷ); `log1p(Price)` is roughly
bell-shaped. *Interpretation:* price is multiplicative, not additive. *ML implication:*
train on `log1p(Price)`, report metrics back on the VND scale via `expm1`.

**Plot 3 — log Price vs log Area.** *Observation:* a clear positive linear band with wide
vertical spread. *Interpretation:* area explains a large share of price but far from all
— location and type matter. *ML implication:* `Area` will be a top feature; keep it,
scale it; expect an R² well below 1.

**Plot 4 — median price by property type.** *Observation:* apartments / townhouses vs
land vs warehouse differ several-fold. *Interpretation:* `Property Type` is a strong
categorical signal and the segments have different price dynamics. *ML implication:*
one-hot encode it; tree models will split on it early.

**Plot 5 — correlation heatmap.** *Observation:* `price_log` correlates most with `Area`
(TODO: fill r), `Width`, room counts; no |r| ≈ 1 among inputs. *Interpretation:* mild
collinearity only. *ML implication:* no feature dropped for redundancy; linear models safe.

## 11. Feature types  (classify every column)

In [ ]:
feature_types = {
    # --- numerical ---
    'Area': 'numerical', 'Width': 'numerical', 'Length': 'numerical',
    'Bedrooms': 'numerical (count)', 'Bathrooms': 'numerical (count)',
    'Floors': 'numerical (count)', 'Alley Width': 'numerical',
    'Agent Listing Count': 'numerical (count)',
    # --- categorical (one-hot) ---
    'Property Type': 'categorical', 'Position': 'categorical',
    'Direction': 'categorical', 'Road Type': 'categorical',
    'Province': 'categorical (63 levels)', 'Agent Role': 'categorical',
    # --- engineered from text (section 13) ---
    'Location': 'text -> parsed into ward / district, then dropped',
    # --- target ---
    'Price': 'target (continuous, million VND)',
    # --- dropped ---
    'Title': 'DROP — free text, leaks Price',
    'Description': 'DROP — free text, not used',
    'Listing ID': 'DROP — identifier (used only for dedupe)',
    'VIP Account': 'DROP — constant',
    'Avatar': 'DROP — not informative',
    'Agent Name': 'DROP — 28k unique, high-cardinality noise',
    'Property Type Slug': 'DROP — duplicate of Property Type',
    'Last Updated': 'DROP — free-text recency string',
    'Scraped At': 'DROP — scrape timestamp',
    'Last Updated Date': 'DROP — timestamp',
    'Latitude': 'DROP — 68% missing', 'Longitude': 'DROP — 68% missing',
}
pd.Series(feature_types, name='role')

**Result.** 8 numerical inputs, 6 one-hot categoricals, 1 text column parsed into 2
engineered categoricals (`ward`, `district`), 1 continuous target, 13 columns dropped
(identifiers, constants, free text, duplicates, sparse coords). Nothing dropped for
redundancy except `Property Type Slug` (identical information to `Property Type`).

## 12. Data representation  — the core Lecture-02 link

```
CSV  ->  DataFrame  ->  clean feature frame  ->  ColumnTransformer (impute + scale + one-hot)  ->  X  ->  model
```

The cell below prints **one raw listing and the feature vector it becomes**, the
DataFrame shape, the final `X` shape and dtype. These exact numbers go into the report's
Mandatory Data-Representation Summary.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# minimal engineered frame just for this demo (full version in section 13)
NUM_DEMO = ['Area', 'Width', 'Length', 'Bedrooms', 'Bathrooms', 'Floors', 'Alley Width']
CAT_DEMO = ['Property Type', 'Position', 'Direction', 'Road Type', 'Province', 'Agent Role']

demo_df = df[NUM_DEMO + CAT_DEMO].copy()
for c in CAT_DEMO:
    demo_df[c] = demo_df[c].fillna('Unknown')
y_demo = np.log1p(df['Price'].values)   # model target = log1p(Price)

demo_prep = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), NUM_DEMO),
    ('cat', OneHotEncoder(handle_unknown='ignore', min_frequency=20), CAT_DEMO),
])
X_demo = demo_prep.fit_transform(demo_df)

print('one RAW listing (selected columns):')
print(df.iloc[0][['Title', 'Price', 'Area', 'Property Type', 'Location']].to_dict())
print('\nsame listing as a FEATURE VECTOR x (first 15 dims of the imputed+scaled+one-hot row):')
row0 = X_demo[0].toarray().ravel() if hasattr(X_demo, 'toarray') else X_demo[0]
print(np.round(row0[:15], 3))
print('\nDataFrame shape at this point :', df.shape, ' (after sections 7-8 cleaning)')
print('feature matrix X shape   :', X_demo.shape, '  # X in R^{N x d}')
print('target vector y shape    :', y_demo.shape, '  # y in R^N  (log1p million VND)')
print('dtype of X               :', X_demo.dtype)
print('one API request is       :  R^{1 x d}   (a single listing)')

**"Model input" in words.** `X` is the array passed to `model.fit(X, y)` /
`model.predict(X)`. Shape `(N, d)`: `N` listings in the batch, `d` features **after**
median-imputing the numeric columns, standardising them, and one-hot encoding the six
categoricals (rare levels folded into `infrequent` by `min_frequency`). It differs from
the raw CSV: the target and all dropped columns are gone, missing values are filled,
categoricals became 0/1 indicator columns, numerics are mean-0/std-1. `y` is
`log1p(Price)` — predictions are converted back with `expm1`. `d` is finalised in
section 13; **TODO: record the exact `d` here after running.**

## 13. Feature engineering

In [ ]:
def parse_district(loc):
    """'Đường X,Phường Y,Quận Z,Tỉnh(Mới)' -> district.
    Prefer a Quận/Huyện/Thị xã/Thành phố token; fall back to the ward
    (Phường/Xã/Thị trấn/Đặc khu); 'Unknown' if neither is present.
    Ward itself is NOT used as a feature: ~4,600 levels is mostly noise."""
    if not isinstance(loc, str):
        return 'Unknown'
    parts = [p.strip() for p in loc.replace('(Mới)', '').split(',') if p.strip()]
    dist = next((p for p in parts if p.startswith(('Quận', 'Huyện', 'Thị xã', 'Thành phố', 'TP.', 'TP '))), None)
    if dist:
        return dist
    return next((p for p in parts if p.startswith(('Phường', 'Xã', 'Thị trấn', 'Đặc khu'))), 'Unknown')


def engineer(frame):
    out = frame.copy()
    # 1. parsed location (district level only)
    out['district'] = out['Location'].apply(parse_district)
    # 2. missing-indicator flags for the >50%-missing counts
    for c in ['Bedrooms', 'Bathrooms', 'Floors']:
        out[c + '_missing'] = out[c].isna().astype(int)
    # 3. cap extreme size (99th pct) so distance/linear models aren't dominated
    for c in ['Area', 'Width', 'Length', 'Alley Width']:
        cap = out[c].quantile(0.99)
        out[c] = out[c].clip(upper=cap)
    # 4. fill categorical NaN with an explicit level
    for c in ['Position', 'Direction', 'Road Type']:
        out[c] = out[c].fillna('Unknown')
    return out


df_eng = engineer(df)

NUM_FEATURES = ['Area', 'Width', 'Length', 'Bedrooms', 'Bathrooms', 'Floors',
                'Alley Width', 'Agent Listing Count',
                'Bedrooms_missing', 'Bathrooms_missing', 'Floors_missing']
CAT_FEATURES = ['Property Type', 'Position', 'Direction', 'Road Type',
                'Province', 'Agent Role', 'district']
FEATURES = NUM_FEATURES + CAT_FEATURES

print('final raw feature count :', len(FEATURES))
print('numeric  :', NUM_FEATURES)
print('categoric:', CAT_FEATURES)
print('district levels        :', df_eng['district'].nunique(),
      '(rare ones folded by min_frequency in section 15)')
df_eng[FEATURES].head(3)

**Justification.**

| Engineered feature | Why |
|---|---|
| `district` (from `Location`) | location is the second-biggest price driver; the raw string is unusable. District is the useful admin level (~2,000 raw levels); `ward` (~4,600) is dropped as noise. One-hot with `min_frequency=50` folds the long tail so `d` stays bounded. |
| `Bedrooms_missing` / `Bathrooms_missing` / `Floors_missing` | these columns are >70% missing and absence correlates with property type (land listings). The flag lets the model use "was this recorded?" instead of trusting the imputed median. |
| 99th-percentile cap on `Area` / `Width` / `Length` / `Alley Width` | a handful of 5,000 m² lots give linear / KNN models huge leverage; capping keeps every row while bounding influence. |

**Encoding choice.** All six original categoricals + the two parsed ones use **one-hot**
(`OneHotEncoder(handle_unknown='ignore', min_frequency=...)`). One-hot, not ordinal,
because none of these have a natural order. `Province` (63) and `district` (hundreds)
rely on `min_frequency` to cap the column count. **Final `d` = TODO (print in section 15).**

## 14. Train / validation / test split  ($D = D_\text{train} \cup D_\text{val} \cup D_\text{test}$)

In [ ]:
from sklearn.model_selection import train_test_split

X_all = df_eng[FEATURES].copy()
y_all = np.log1p(df_eng['Price'].values)          # model target
y_all_vnd = df_eng['Price'].values                # kept for reporting on the real scale

X_train, X_tmp, y_train, y_tmp, _, y_tmp_vnd = train_test_split(
    X_all, y_all, y_all_vnd, test_size=0.30, random_state=RANDOM_SEED)
X_val, X_test, y_val, y_test, y_val_vnd, y_test_vnd = train_test_split(
    X_tmp, y_tmp, y_tmp_vnd, test_size=0.50, random_state=RANDOM_SEED)

print(f'train: {X_train.shape}   val: {X_val.shape}   test: {X_test.shape}   (~70/15/15)')

**Why test data must not influence training or preprocessing fitting (data leakage).**
If the imputer's medians, the scaler's mean/std, or the one-hot vocabulary were computed
over rows that later appear in validation/test, the model would indirectly "know" those
rows and the reported error would be optimistically low — it would not hold on genuinely
new listings. So the split happens **before** any `.fit()`, and every preprocessing step
is fitted on `X_train` only (section 15). Regression needs no stratification, but the
`Listing ID` dedupe in section 7 already prevents the *same* property landing on both
sides. `random_state` fixed for reproducibility.

## 15. Preprocessing pipeline  (ONE object, fitted on train only, later persisted)

In [ ]:
numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=50, sparse_output=True)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, NUM_FEATURES),
    ('cat', categorical_pipe, CAT_FEATURES),
])

preprocessor.fit(X_train)                    # <-- fitted on TRAIN ONLY
X_train_pp = preprocessor.transform(X_train)
D_FEATURES = X_train_pp.shape[1]
print('preprocessed train shape:', X_train_pp.shape, '  # d =', D_FEATURES)
print('(numeric', len(NUM_FEATURES), '+ one-hot', D_FEATURES - len(NUM_FEATURES),
      'indicator columns, rare levels folded into an <infrequent> column)')

**Steps and purpose.** (1) numeric: median-impute the missing sizes/counts, then
`StandardScaler` so linear regression, SVR and KNN treat every feature on a comparable
scale; (2) categorical: fill `'Unknown'`, then one-hot with `min_frequency=25` so rare
wards/districts/provinces collapse into an `infrequent` column instead of exploding `d`.
This one fitted `preprocessor` is bundled with the chosen model in section 22 and loaded
**unchanged** by `../api/` — deployment never re-fits it.

## 16. Baseline model  (a score to beat)

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

LOG_CLIP = np.log1p(1_000_000)   # cap predictions at 1,000,000 million VND before expm1

def _to_vnd(y_log):
    """Invert the log1p target, guarding against overflow from wild linear extrapolation."""
    return np.expm1(np.clip(y_log, None, LOG_CLIP))

def regression_scores(y_true_log, y_pred_log):
    """Scores reported on the real VND scale (million VND)."""
    yt, yp = _to_vnd(y_true_log), _to_vnd(y_pred_log)
    mae = mean_absolute_error(yt, yp)
    mse = mean_squared_error(yt, yp)
    rmse = np.sqrt(mse)
    r2 = r2_score(yt, yp)
    mape = np.mean(np.abs((yt - yp) / np.clip(yt, 1e-6, None))) * 100
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2, 'MAPE_%': mape}

base = Pipeline([('prep', preprocessor), ('reg', DummyRegressor(strategy='median'))])
base.fit(X_train, y_train)
base_scores = regression_scores(y_val, base.predict(X_val))
print('baseline (predict median log-price):')
for k, v in base_scores.items():
    print(f'  {k:7s}: {v:,.3f}')

**Reference score.** Always predicting the median log-price gives MAE ≈ TODO million
VND and R² ≈ 0 on validation. Every trained model below must beat this — a model that
cannot is worse than a constant guess.

## 17. Model training  (5 regression models, same preprocessed train data)

Appendix (§9.6) requires at least four; we compare **five**: Linear Regression, Ridge,
Decision Tree, Random Forest, Gradient Boosting (XGBoost). All share the identical
`preprocessor` and `X_train` / `y_train` (log target).

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
try:
    from xgboost import XGBRegressor
    _HAS_XGB = True
except ImportError:
    from sklearn.ensemble import GradientBoostingRegressor
    _HAS_XGB = False

# Train all models on the SAME sample so the comparison is fair and the notebook
# runs in a few minutes. Raise / remove TRAIN_SAMPLE for a final run.
TRAIN_SAMPLE = 50_000
if len(X_train) > TRAIN_SAMPLE:
    samp_idx = np.random.RandomState(RANDOM_SEED).choice(len(X_train), TRAIN_SAMPLE, replace=False)
    X_tr_s, y_tr_s = X_train.iloc[samp_idx], y_train[samp_idx]
else:
    X_tr_s, y_tr_s = X_train, y_train
print(f'training sample: {X_tr_s.shape[0]:,} rows')

models = {
    'LinearRegression': LinearRegression(),
    'Ridge'           : Ridge(alpha=10.0, random_state=RANDOM_SEED),
    'DecisionTree'    : DecisionTreeRegressor(max_depth=12, min_samples_leaf=25, random_state=RANDOM_SEED),
    'RandomForest'    : RandomForestRegressor(n_estimators=120, max_depth=18, min_samples_leaf=10,
                                              n_jobs=-1, random_state=RANDOM_SEED),
}
if _HAS_XGB:
    models['XGBoost'] = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                     subsample=0.8, colsample_bytree=0.8, tree_method='hist',
                                     objective='reg:squarederror', random_state=RANDOM_SEED, n_jobs=-1)
else:
    models['GradientBoosting'] = GradientBoostingRegressor(random_state=RANDOM_SEED)

fitted, timings = {}, {}
for name, reg in models.items():
    pipe = Pipeline([('prep', preprocessor), ('reg', reg)])
    t0 = time.time(); pipe.fit(X_tr_s, y_tr_s); dt = time.time() - t0
    fitted[name] = pipe; timings[name] = dt
    print(f'{name:18s} trained in {dt:6.2f}s')

**Note.** Tree models (`DecisionTree`, `RandomForest`, `XGBoost`) don't need the scaling,
but running them through the same `preprocessor` keeps one code path and one persisted
object. `max_depth` / `min_samples_leaf` are set conservatively to limit overfitting on
220k rows; **TODO: light hyper-parameter search if time permits, then note it here.**

## 18. Model comparison  (validation set)

In [ ]:
rows = []
for name, pipe in fitted.items():
    s = regression_scores(y_val, pipe.predict(X_val))
    s['model'] = name; s['train_s'] = round(timings[name], 2)
    rows.append(s)
cmp = pd.DataFrame(rows).set_index('model')[['MAE', 'MSE', 'RMSE', 'R2', 'MAPE_%', 'train_s']]
cmp = cmp.sort_values('RMSE')
cmp.round(3)

**Which model leads.** *(Fill after running.)* Expect the gradient-boosted model and
Random Forest to give the lowest RMSE / highest R², Linear and Ridge to trail (they
cannot capture area×location interactions), and the single Decision Tree to sit between.
The best by validation RMSE is carried to the held-out test set. **TODO: name the winner
and quote its MAE (million VND) and R².**

## 19. Evaluation  (chosen model, HELD-OUT test set)

In [ ]:
BEST = cmp.index[0]          # lowest validation RMSE  (TODO: confirm / override)
print('selected model:', BEST)

final_pipe = fitted[BEST]
test_scores = regression_scores(y_test, final_pipe.predict(X_test))
for k, v in test_scores.items(): print(f'  {k:7s}: {v:,.3f}')

# predicted vs actual (VND scale)
yp = np.expm1(final_pipe.predict(X_test)); yt = y_test_vnd
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(yt, yp, s=6, alpha=0.3); lim = [0, np.percentile(yt, 99)]
ax[0].plot(lim, lim, 'r--'); ax[0].set_xlim(lim); ax[0].set_ylim(lim)
ax[0].set_xlabel('actual Price (million VND)'); ax[0].set_ylabel('predicted'); ax[0].set_title('Predicted vs Actual')
resid = yp - yt
ax[1].scatter(yt, resid, s=6, alpha=0.3); ax[1].axhline(0, color='r', ls='--')
ax[1].set_xlim(lim); ax[1].set_xlabel('actual Price'); ax[1].set_ylabel('residual (pred - actual)')
ax[1].set_title('Residuals vs Actual')
plt.tight_layout(); plt.show()

**Metrics interpreted (what each means for this application).**

- **MAE** — the average absolute miss in **million VND**. "On a typical listing the
  estimate is off by ~X tỷ." The number a user feels.
- **MSE / RMSE** — squared-error based, so large misses (mispricing a villa) are punished
  far more than small ones. RMSE is back on the VND scale and is the headline error.
- **R²** — share of price variance the model explains vs. always guessing the mean
  (1 = perfect, 0 = no better than the mean). TODO: quote it.
- **MAPE** — average *percentage* error; comparable across cheap and expensive listings.

*(No confusion matrix — this is regression, not classification.)* The predicted-vs-actual
plot should hug the diagonal; the residual plot should be a flat band around 0. TODO:
note any funnel shape (heteroscedasticity — bigger errors on pricier homes).

## 20. Error analysis  (where it fails)

In [ ]:
err = X_test.copy()
err['actual'] = y_test_vnd
err['pred'] = np.expm1(final_pipe.predict(X_test))
err['abs_err'] = (err['pred'] - err['actual']).abs()
err['pct_err'] = err['abs_err'] / err['actual'] * 100

print('worst 10 absolute errors:')
display(err.sort_values('abs_err', ascending=False).head(10)[
    ['Property Type', 'Province', 'district', 'Area', 'actual', 'pred', 'abs_err', 'pct_err']])

print('\nmean |%err| by property type:')
display(err.groupby('Property Type')['pct_err'].mean().sort_values(ascending=False).round(1))

print('\nmean |%err| by area band:')
err['area_band'] = pd.cut(err['Area'], [0, 50, 100, 200, 500, 10000])
display(err.groupby('area_band')['pct_err'].mean().round(1))

**What the model struggles with.** *(Fill after running.)* Typically: (i) very large or
very cheap listings — thin training data at the extremes; (ii) one or two property types
(e.g. `Kho, nhà xưởng`) with few rows; (iii) districts folded into the `infrequent`
one-hot bucket, which lose their local price level. Likely causes: representation gap
(no true street-level location, no year-built / legal-status), skewed target, sparse
segments. Possible fixes: target-encode `district`, add missing attributes, collect more
rows for rare types, or a separate model per property type.

## 21. Model selection  (markdown — justify the deployed model)

**Deployed model: TODO (the section 18/19 winner, likely XGBoost or RandomForest).**

| Criterion | Chosen model | Note |
|---|---|---|
| Predictive performance | lowest test RMSE + highest R² | primary reason |
| Interpretability | feature importances available (tree/boosting) | can show "area, district, type drove this estimate" |
| Computational cost | trains in ~TODO s, predicts in &lt;10 ms/listing | fine for a request/response API |
| Robustness | ensemble handles the skew + mild outliers left after section 8 | |
| Deployment constraints | ~TODO MB pickled, CPU-only, no GPU needed | container-friendly |

Linear Regression / Ridge are kept as a documented fallback (smaller, fully
interpretable, but clearly higher error).

## 22. Model persistence  (save the full inference artifact)

In [ ]:
import joblib, os, json

os.makedirs('../model', exist_ok=True)

# refit the chosen pipeline on train + val so deployment uses all non-test data
deploy_pipe = Pipeline([('prep', preprocessor), ('reg', models[BEST])])
# refit on the training sample + validation set (consistent with section 17)
X_fit = pd.concat([X_tr_s, X_val]); y_fit = np.concatenate([y_tr_s, y_val])
deploy_pipe.fit(X_fit, y_fit)

joblib.dump(deploy_pipe, '../model/model_pipeline.joblib')
joblib.dump(FEATURES,    '../model/feature_names.joblib')

schema = {
    'target': 'Price (million VND); model trained on log1p(Price), invert with expm1',
    'task': 'regression',
    'chosen_model': BEST,
    'random_seed': RANDOM_SEED,
    'sklearn_version': sklearn.__version__,
    'price_unit': 'million VND',
    'numeric_features': NUM_FEATURES,
    'categorical_features': CAT_FEATURES,
    'model_features_order': FEATURES,
    'engineered': ['ward', 'district (parsed from Location)',
                   'Bedrooms_missing', 'Bathrooms_missing', 'Floors_missing',
                   '99th-pct cap on Area/Width/Length/Alley Width'],
    'dropped_columns': ['Title (leaks Price)', 'Description', 'Listing ID', 'VIP Account',
                        'Avatar', 'Agent Name', 'Property Type Slug', 'Last Updated',
                        'Scraped At', 'Last Updated Date', 'Latitude', 'Longitude', 'Location'],
}
with open('../model/input_schema.json', 'w') as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)

print('saved ../model/model_pipeline.joblib   (preprocessing + regressor in one object)')
print('saved ../model/feature_names.joblib    (expected raw input columns, ordered)')
print('saved ../model/input_schema.json       (input contract for ../api/)')

**Files produced.** `model/model_pipeline.joblib` — one object holding the fitted
imputers + scaler + one-hot encoder + the chosen regressor. `model/feature_names.joblib`
— the ordered raw column list `../api/` validates against. `model/input_schema.json` —
the human-readable contract. Nothing else is needed at inference; the API loads exactly
these and re-fits nothing.

## 23. Inference test  (reload from disk, ONE raw dict, full path -> JSON)

In [ ]:
loaded_pipe     = joblib.load('../model/model_pipeline.joblib')
loaded_features = joblib.load('../model/feature_names.joblib')

raw_listing = {          # exactly what the API / mobile client sends (raw, unprocessed)
    'Area': 78.7, 'Width': 4.0, 'Length': np.nan,
    'Bedrooms': 3, 'Bathrooms': 2, 'Floors': 2, 'Alley Width': np.nan,
    'Agent Listing Count': 2,
    'Property Type': 'Nhà riêng', 'Position': 'Đường chính', 'Direction': 'Nam',
    'Road Type': 'Đường nhựa', 'Province': 'an-giang', 'Agent Role': 'Chính chủ',
    'ward': 'Phường An Hòa', 'district': 'Rạch Giá',
}
# the engineered missing-flags the client can't know — derive them the same way the notebook did
for c in ['Bedrooms', 'Bathrooms', 'Floors']:
    raw_listing[c + '_missing'] = int(pd.isna(raw_listing.get(c)))

row = pd.DataFrame([raw_listing])[loaded_features]     # right columns, right order
pred_log = float(loaded_pipe.predict(row)[0])
pred_price = float(np.expm1(pred_log))                 # back to million VND

result = {
    'predicted_price': round(pred_price, 2),
    'price_per_m2': round(pred_price * 1000 / raw_listing['Area'], 2),
    'currency': 'million VND',
    'model': BEST,
}
print(result)

**Contract confirmed.** The reloaded pipeline (fresh from disk) turns a **raw dict**
into `{ "predicted_price": <million VND>, "price_per_m2": ..., "model": ... }` via
`raw input -> validate columns -> the same fitted preprocessing -> regressor -> expm1`.
**No new scaler / imputer / encoder was fitted here.** This is exactly what `../api/`
does — `POST /predict` just wraps this call (Appendix C).

---
## Mandatory Data-Representation Summary  (copy into the report)

| Item | Value |
|---|---|
| Raw form | CSV / tabular, 1 row = 1 property listing |
| Rows after cleaning (`N`) | TODO (print `len(df_eng)`) |
| Raw columns | 28 (1 target + 27 candidates) |
| Features used before encoding | 18 (11 numeric incl. 3 missing-flags, 7 categorical) |
| Numerical representation | median-impute → `StandardScaler` (numeric); `'Unknown'` → one-hot `min_frequency=25` (categorical) |
| Final feature dimension `d` | TODO (print `X_train_pp.shape[1]`) |
| Model input shape | `X_train ∈ ℝ^{TODO_N × TODO_d}`; one request = `ℝ^{1 × TODO_d}` |
| Target | `y = log1p(Price) ∈ ℝ^N`; reported back as `expm1(y)` in million VND |
| dtype | `float64` (sparse CSR after one-hot) |

**Every dimension above must be explained in the report** — `N` = listings kept after
dedupe + domain filtering, `d` = numeric columns + one-hot indicator columns (rare levels
folded).